In [3]:
# ============================================================
# 셀 1 : 라이브러리 임포트
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

import joblib

print("라이브러리 로드 완료")


# ============================================================
# 셀 2 : 설정값 (경로·컬럼·가중치)
# ============================================================

FILE_PATH     = "하나투어.xlsx"
FEEDBACK_FILE = "user_feedback.csv"

FEATURES = [
    "방문국가",
    "연령",
    "동반자유형",
    "여행목적",
    "여행일수",
    "패키지여부",
]

CATEGORICAL = [
    "방문국가",
    "연령",
    "동반자유형",
    "여행목적",
    "패키지여부",
]

WEIGHTS = {
    "연령":     20,
    "여행목적": 25,
    "동반자유형": 25,
    "여행일수": 15,
    "1인당비용": 15,
}

REQUIRED_COLS = [
    "방문국가", "연령", "동반자유형",
    "여행목적", "여행일수", "패키지여부",
    "1인당비용", "만족도", "숙박시설",
]

print("설정값 로드 완료")


# ============================================================
# 셀 3 : 데이터 로드 및 전처리
# ============================================================

df = pd.read_excel(FILE_PATH)
df = df.dropna(subset=REQUIRED_COLS)

print(f"데이터 로드 완료 | shape: {df.shape}")
print("컬럼 목록:", df.columns.tolist())


# ============================================================
# 셀 4 : 전처리 파이프라인 & 모델 학습
# ============================================================

PREPROCESSOR = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)
    ],
    remainder="passthrough",
)

# --- 만족도 예측 (회귀) ---
satisfaction_model = Pipeline([
    ("prep",  PREPROCESSOR),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42)),
])
satisfaction_model.fit(df[FEATURES], df["만족도"])
print("만족도 모델 학습 완료")

# --- 1인당 비용 예측 (회귀) ---
cost_model = Pipeline([
    ("prep",  PREPROCESSOR),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42)),
])
cost_model.fit(df[FEATURES], df["1인당비용"])
print("비용 모델 학습 완료")

# --- 숙박시설 예측 (분류) ---
hotel_model = Pipeline([
    ("prep",  PREPROCESSOR),
    ("model", RandomForestClassifier(n_estimators=300, random_state=42)),
])
hotel_model.fit(df[FEATURES], df["숙박시설"])
print("숙박시설 모델 학습 완료")


# ============================================================
# 셀 5 : 유사도 점수 계산 함수
# ============================================================

def calculate_score(user: dict, row: pd.Series) -> float:
    """사용자 입력과 데이터 행 간의 적합도 점수(0~100)를 반환합니다."""
    score, total = 0.0, 0.0

    # 범주형 일치 여부
    for col in ["연령", "여행목적", "동반자유형"]:
        if user.get(col):
            total += WEIGHTS[col]
            if user[col] == row[col]:
                score += WEIGHTS[col]

    # 여행일수 : 차이 1일당 2점 감점
    if user.get("여행일수"):
        total += WEIGHTS["여행일수"]
        diff = abs(float(user["여행일수"]) - float(row["여행일수"]))
        score += max(0.0, WEIGHTS["여행일수"] - diff * 2)

    # 1인당비용 : 비율 차이에 비례하여 감점
    if user.get("1인당비용"):
        total += WEIGHTS["1인당비용"]
        diff_ratio = abs(
            float(user["1인당비용"]) - float(row["1인당비용"])
        ) / max(float(user["1인당비용"]), 1)
        score += max(0.0, WEIGHTS["1인당비용"] * (1 - diff_ratio))

    return round((score / total) * 100, 2) if total > 0 else 0.0


# ============================================================
# 셀 6 : 추천 함수
# ============================================================

def recommend(user: dict) -> list[dict]:
    """
    user 딕셔너리를 받아 상위 3개 여행 케이스를 반환합니다.

    필수 키 : 방문국가
    선택 키 : 연령, 여행목적, 동반자유형, 여행일수, 1인당비용, 패키지선택
              (선택 항목 중 최소 2개 이상 필요)
    """

    # ── 입력 검증 ──────────────────────────────────────────
    if not user.get("방문국가"):
        raise ValueError("국가는 필수입니다.")

    optional_filled = sum(
        bool(user.get(c))
        for c in ["연령", "여행목적", "동반자유형", "여행일수", "1인당비용"]
    )
    if optional_filled < 2:
        raise ValueError("국가 외 최소 2개 이상의 조건을 입력해주세요.")

    # ── 국가 필터 ──────────────────────────────────────────
    data = df[df["방문국가"] == user["방문국가"]].copy()

    # ── 패키지 필터 ────────────────────────────────────────
    travel_type = user.get("패키지선택", "상관없음")
    if travel_type == "패키지":
        data = data[data["패키지여부"].astype(str).str.contains("구매|패키지")]
    elif travel_type == "자유여행":
        data = data[~data["패키지여부"].astype(str).str.contains("구매|패키지")]

    if data.empty:
        return [{"message": "조건에 맞는 데이터가 없습니다."}]

    # ── 적합도 점수 계산 & 상위 3건 추출 ──────────────────
    data["score"] = data.apply(lambda row: calculate_score(user, row), axis=1)
    top3 = data.nlargest(3, "score")

    # ── 결과 구성 ──────────────────────────────────────────
    results = []
    country_data = df[df["방문국가"] == user["방문국가"]]
    avg_cost     = country_data["1인당비용"].mean()

    cheapest  = country_data.loc[country_data["1인당비용"].idxmin()]
    expensive = country_data.loc[country_data["1인당비용"].idxmax()]
    average   = country_data.iloc[
        (country_data["1인당비용"] - avg_cost).abs().argsort()[:1]
    ].iloc[0]

    for rank, (_, row) in enumerate(top3.iterrows(), start=1):
        ml_input = pd.DataFrame([{
            "방문국가":   row["방문국가"],
            "연령":       row["연령"],
            "동반자유형": row["동반자유형"],
            "여행목적":   row["여행목적"],
            "여행일수":   row["여행일수"],
            "패키지여부": row["패키지여부"],
        }])

        results.append({
            "순위":       rank,
            "국가":       row["방문국가"],
            "적합도":     row["score"],
            "예상만족도": round(float(satisfaction_model.predict(ml_input)[0]), 2),
            "예상비용":   int(cost_model.predict(ml_input)[0]),
            "추천숙박":   str(hotel_model.predict(ml_input)[0]),
            "최저가사례": {
                "비용":   int(cheapest["1인당비용"]),
                "일수":   int(cheapest["여행일수"]),
                "만족도": float(cheapest["만족도"]),
            },
            "평균사례": {
                "비용":   int(average["1인당비용"]),
                "일수":   int(average["여행일수"]),
                "만족도": float(average["만족도"]),
            },
            "최고가사례": {
                "비용":   int(expensive["1인당비용"]),
                "일수":   int(expensive["여행일수"]),
                "만족도": float(expensive["만족도"]),
            },
        })

    return results


# ============================================================
# 셀 7 : 추천 테스트 실행
# ============================================================

user_input = {
    "방문국가":   "일본",
    "연령":       "30대",
    "여행목적":   "휴양",
    "동반자유형": "배우자",
    "여행일수":   5,
    "1인당비용":  1_500_000,
    "패키지선택": "상관없음",   # "패키지" | "자유여행" | "상관없음"
}

results = recommend(user_input)

print("\n===== 추천 결과 =====\n")
for item in results:
    print(item)


# ============================================================
# 셀 8 : 피드백 저장 함수
# ============================================================

def save_feedback(feedback_data: dict) -> None:
    """사용자 후기를 CSV 파일에 누적 저장합니다."""
    row = pd.DataFrame([feedback_data])

    if os.path.exists(FEEDBACK_FILE):
        row.to_csv(FEEDBACK_FILE, mode="a", header=False,
                   index=False, encoding="utf-8-sig")
    else:
        row.to_csv(FEEDBACK_FILE, index=False, encoding="utf-8-sig")

    print("피드백 저장 완료")


# 예시 후기 저장
save_feedback({
    "방문국가":   "일본",
    "연령":       "30대",
    "동반자유형": "배우자",
    "여행목적":   "휴양",
    "여행일수":   5,
    "패키지여부": "패키지",
    "1인당비용":  1_700_000,
    "숙박시설":   "리조트",
    "만족도":     4.2,
})


# ============================================================
# 셀 9 : 모델 재학습 함수
# ============================================================

def load_all_data() -> pd.DataFrame:
    """원본 데이터와 피드백 데이터를 합쳐 반환합니다."""
    original = pd.read_excel(FILE_PATH)
    if os.path.exists(FEEDBACK_FILE):
        feedback = pd.read_csv(FEEDBACK_FILE)
        original = pd.concat([original, feedback], ignore_index=True)
    return original


def retrain_models() -> None:
    """전체 데이터로 세 모델을 재학습하고 pkl 파일로 저장합니다."""
    global df, satisfaction_model, cost_model, hotel_model

    print("재학습 시작...")
    df = load_all_data().dropna(subset=REQUIRED_COLS)

    X = df[FEATURES]
    satisfaction_model.fit(X, df["만족도"])
    cost_model.fit(X, df["1인당비용"])
    hotel_model.fit(X, df["숙박시설"])

    joblib.dump(satisfaction_model, "satisfaction_model.pkl")
    joblib.dump(cost_model,         "cost_model.pkl")
    joblib.dump(hotel_model,        "hotel_model.pkl")

    print(f"총 학습 데이터 수 : {len(df)}건")
    print("재학습 완료")


# ============================================================
# 셀 10 : 자동 재학습 (피드백 100건 이상 시 실행)
# ============================================================

def auto_retrain(threshold: int = 100) -> None:
    """피드백이 threshold 건 이상 쌓이면 자동으로 재학습합니다."""
    if not os.path.exists(FEEDBACK_FILE):
        print("피드백 데이터 없음 → 재학습 건너뜀")
        return

    feedback_count = len(pd.read_csv(FEEDBACK_FILE))
    print(f"누적 피드백 수 : {feedback_count}건")

    if feedback_count >= threshold:
        print(f"{threshold}건 이상 누적 → 재학습 실행")
        retrain_models()
    else:
        print(f"재학습 조건 미달 ({feedback_count}/{threshold})")


auto_retrain()


라이브러리 로드 완료
설정값 로드 완료
데이터 로드 완료 | shape: (2157, 12)
컬럼 목록: ['성별', '연령', '방문국가', '총여행비', '1인당비용', '여행일수', '동반자수', '동반자유형', '패키지여부', '여행목적', '만족도', '숙박시설']
만족도 모델 학습 완료
비용 모델 학습 완료
숙박시설 모델 학습 완료

===== 추천 결과 =====

{'순위': 1, '국가': '일본', '적합도': 71.0, '예상만족도': 3.77, '예상비용': 1158647, '추천숙박': '고급숙박', '최저가사례': {'비용': 30000, '일수': 3, '만족도': 4.0}, '평균사례': {'비용': 770000, '일수': 3, '만족도': 4.5}, '최고가사례': {'비용': 5250000, '일수': 7, '만족도': 4.0}}
{'순위': 2, '국가': '일본', '적합도': 69.8, '예상만족도': 4.24, '예상비용': 830865, '추천숙박': '고급숙박', '최저가사례': {'비용': 30000, '일수': 3, '만족도': 4.0}, '평균사례': {'비용': 770000, '일수': 3, '만족도': 4.5}, '최고가사례': {'비용': 5250000, '일수': 7, '만족도': 4.0}}
{'순위': 3, '국가': '일본', '적합도': 69.6, '예상만족도': 4.44, '예상비용': 1050126, '추천숙박': '고급숙박', '최저가사례': {'비용': 30000, '일수': 3, '만족도': 4.0}, '평균사례': {'비용': 770000, '일수': 3, '만족도': 4.5}, '최고가사례': {'비용': 5250000, '일수': 7, '만족도': 4.0}}
피드백 저장 완료
누적 피드백 수 : 1건
재학습 조건 미달 (1/100)
